In [ ]:
# settings on osc gpu
!module load cuda
!module load pytorch

In [ ]:
import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
if torch.cuda.is_available():
    print(torch.cuda.get_device_properties(0))

In [ ]:
# doc: torch.unsqueeze()
# https://pytorch.org/docs/stable/torch.html#torch.unsqueeze
# doc: torch. squeeze()
# https://pytorch.org/docs/stable/torch.html#torch.squeeze

def make_features(x):
    """build features i.e. a matrix with columns [x, x^2, x^3]"""
    x=x.unsqueeze(1)
    return torch.cat([x**i for i in range(1,4)],1)

def f(x):
    w_target = torch.FloatTensor([0.5,3,2.4]).unsqueeze(1)
    b_target = torch.FloatTensor([0.9])
    return x.mm(w_target)+b_target

def get_batch(batch_size=32):
    """builds a batch i.e. (x, f(x)) pairs"""
    random = torch.randn(batch_size)
    x = make_features(random)
    y = f(x)
    if torch.cuda.is_available:
        return Variable(x).cuda(), Variable(y).cuda()
    else:
        return Variable(x), Variable(y)

In [ ]:
# define model
class PolyRegression(nn.Module):
    def __init__(self):
        super(PolyRegression,self).__init__()
        self.model=nn.Linear(3,1)
    
    def forward(self, x):
        return self.model(x)

# init model
if torch.cuda.is_available:
    model=PolyRegression().cuda()
else:
    model=PolyRegression()

In [ ]:
num_epochs=10**3
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(),lr=1e-3)
for epoch in range(num_epochs):
    # forward
    batch_x, batch_y = get_batch()
    output = model(batch_x)
    loss = criterion(output, batch_y)
    #backword
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % (num_epochs/10) == 0:
        print('Epoch {:5d}/{:5d}, loss = {:.6f}'.format((epoch+1), num_epochs, loss.data))

In [ ]:
# print results
model.eval()
batch_x1, batch_y1 = get_batch(10**3)
predict = model(batch_x1).cpu()
predict = predict.data.numpy()
batch_x1 = batch_x1.cpu()
batch_x = batch_x.cpu()
batch_y = batch_y.cpu()
# display(batch_x[:,0])
# display(batch_y)
# display(predict)
plt.plot(batch_x1.numpy()[:,0], predict, '.', label='predict')
plt.plot(batch_x.numpy()[:,0], batch_y.numpy(),'ro', label='original data')
plt.legend()
plt.show()